### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following: 

* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detection

In [65]:
import os 
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")    


### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

* Long-running conversation that exceed context windows.
* Multi-turn dialogues with extensive history.
* Application where preserving full conversation context matters.

In [66]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message-based summarization
agent = create_agent(
    model="gpt-5",
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-5",
            trigger=("messages", 5),
            keep=("messages", 4),
        ),
    ]
)

In [67]:
### Run with thread id
config = {"configurable": {"thread_id": "test-1"}}

In [68]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is the capital of France?",
    "Who is the president of the United States?",
    "What is the largest mammal?",
    "What is the speed of light?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='948599a2-0bfe-4940-ba8f-5143fb212b81'), AIMessage(content='4', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj0NSoslPBbGRupOJOeNAqU6Q3Rdx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e5988-7c9a-78f3-9340-1ab041dad1a4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### Token Size

In [69]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver

@tool
def search_hotels(city:str) ->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $300/night, spa, pool, gym, free Wi-Fi
    2. City Inn - 4 star, $200/night, gym, free Wi-Fi
    3. Budget Stay - 3 star, $100/night, free Wi-Fi
    4. Luxury Suites - 5 star, $400/night, spa, pool"""

agent=create_agent(
    model="gpt-5",
    tools=[search_hotels],
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-5",
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximately)
def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # Approximate 4 chars per token

In [70]:
# Run test

cities = ["New York", "Paris", "Tokyo", "Sydney", "Cairo"]
for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}.")]},
        config=config
    )
    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

New York: ~172 tokens, 4 messages
[HumanMessage(content='Find hotels in New York.', additional_kwargs={}, response_metadata={}, id='f18d7b76-e757-4e76-87a0-c44f519d66da'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 409, 'prompt_tokens': 136, 'total_tokens': 545, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj0O9p4RFWRGhP66WHUwUOdqGEWRI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e5989-23f4-7713-9d0d-1465dd6088b5-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'New York'}, 'id': 'call_42TYPEP7hS7NpPjlNVVxHllk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 136,

### Fraction

In [71]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver

@tool
def search_hotels(city:str) ->str:
    """Search hotels - returns long response to use more tokens."""
    return f"Hotels in {city}: Grand Hotel - 5 star, $300/night, City Inn $180, Budget Stay $75"
# Low fraction for testing
agent=create_agent(
    model="gpt-5",
    tools=[search_hotels],
    checkpointer=MemorySaver(),
    middleware=[
         SummarizationMiddleware(
            model="gpt-5",
            trigger=("fraction", 0.005), # 0.5% = ~ 500 tokens
            keep=("fraction", 0.002), # 0.2% = ~ 200 tokens
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximately)
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4  # Approximate 4 chars

# Test
cities = ["New York", "Paris", "Tokyo", "Sydney", "Cairo"]

for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Hotels in {city}.")]},
        config=config
    )
    tokens=count_tokens(response["messages"])
    fraction = tokens / 10000  # Assuming 10k token limit
    print(f"{city}: ~{tokens} tokens ({fraction:.4%} of limit), {len(response['messages'])} messages")
    print(response['messages'])

New York: ~105 tokens (1.0500% of limit), 4 messages
[HumanMessage(content='Hotels in New York.', additional_kwargs={}, response_metadata={}, id='daace8d2-e95b-40a1-976c-1e888ed6e9f4'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 153, 'prompt_tokens': 135, 'total_tokens': 288, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj0Q2gLOlRZA5VOXTKDaZOCZX1rte', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e598a-edd7-7fe3-97ac-3483330f14fc-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'New York'}, 'id': 'call_JcrfSRkJ8TxmSWrJDpfJ4KLI', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input

### Human  In The Loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following: 
* High-stakes operations requiring human approval (e.g. database writes, financial transactions).
* Compliance workflows where human oversight is mandatory.
* Long-running conversation where human feedback guides the agent.

In [72]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver

def read_email_tool(email_id:str) ->str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str) ->str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [73]:
agent = create_agent(
    model="gpt-5",
    tools=[read_email_tool, send_email_tool],
    checkpointer=MemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        ) 
    ]
)

In [74]:
config={"configurable":{"thread_id":"test-1"}} # thread_id is unique id
#Step 1: Request

result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [75]:
result

{'messages': [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='e88dbac1-ba76-42bb-b45d-438af9685f72'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 178, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj0QPzrHFT6MYEqNGFAMW1LlpkVjZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e598b-49c1-7d62-bbb4-9197415d0615-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@example.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_UY7dqSoEnBsewn5uEv

In [76]:
# Step 2: Human approval
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused for human approval...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "approve"
                    }
                ]
            }
        ),
        config=config
    )

    print(result["messages"][-1].content)

Paused for human approval...
Your email has been sent to john@example.com with the subject "Hello" and the body "How are you?".


In [77]:
result

{'messages': [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='e88dbac1-ba76-42bb-b45d-438af9685f72'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 178, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj0QPzrHFT6MYEqNGFAMW1LlpkVjZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e598b-49c1-7d62-bbb4-9197415d0615-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@example.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_UY7dqSoEnBsewn5uEv

### Reject

In [78]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver

def read_email_tool(email_id:str) ->str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str) ->str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'" 

agent = create_agent(
    model="gpt-5",
    tools=[read_email_tool, send_email_tool],
    checkpointer=MemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        ) 
    ]
)

config={"configurable":{"thread_id":"test-reject"}} # thread_id is unique id
#Step 1: Request

result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [79]:
# Step 2: Human approval
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused for human approval...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "reject"
                    }
                ]
            }
        ),
        config=config
    )

    print(result["messages"][-1].content)

Paused for human approval...
I couldn’t send the email because the send action wasn’t authorized.

Would you like me to send it now?  
- To: john@example.com  
- Subject: Hello  
- Body: How are you?

Reply “Send” to proceed, or use this link to send from your mail app: mailto:john@example.com?subject=Hello&body=How%20are%20you%3F


In [80]:
result

{'messages': [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0b80bec4-a3cc-452b-8555-13ae3a41443d'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 178, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj0QT9PPtMbV26axYX1jl0S0uGrJY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e598b-5973-7543-9db8-e82e2e46245f-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@example.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_MMj1TzCjzbMFXGl5dP

### Edit


In [84]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver

def read_email_tool(email_id:str) ->str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str) ->str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'" 

agent = create_agent(
    model="gpt-5",
    tools=[read_email_tool, send_email_tool],
    checkpointer=MemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        ) 
    ]
)

config={"configurable":{"thread_id":"test-reject"}} # thread_id is unique id
#Step 1: Request

result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'How are you?'")]},
    config=config
) 

In [ ]:
# Step 2: Human approval
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused for human approval...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit"
                    }
                ]
            }
        ),
        config=config
    )

    print(result["messages"][-1].content)

In [ ]:
# Step 2: Human approval
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused for human approval...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit"
                    }
                ]
            }
        ),
        config=config
    )

    print(result["messages"][-1].content)

In [86]:
result

{'messages': [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='045ebb01-53bb-4df3-adde-88fb04171a53'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 178, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dj0RWChkgehkDItSFH4kWyixIbm50', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e598c-55d0-7d50-b23d-4d050337f4ca-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@example.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_Ag5UiOvYiK1C6U76tH